# Interactive Visualizer

The simplest way to show a scene in a notebook is the context manager — it clears the scene on entry and calls `show()` (which renders inline) on exit. `viz(...)` is shorthand for `viz.new(...)`.

Under Jupyter, `Visualizer()` is a **singleton**: re-running a cell that re-creates it reuses the same instance (one server, one scene host) and clears the default scene, so every cell below is safe to re-run.

Keywords: interactive, visualizer, context manager, show, display, notebook, re-run, singleton


In [ ]:
from pytanga.geometry import Direction, Plane, Point, Sphere
from pytanga.viz import SphereStyle, Visualizer

## The context manager (simplest)


In [ ]:
with Visualizer() as viz:  # clear + show on entry, flush on exit
    viz(Point(1, 2, 3), color="#ff4444")
    viz(Sphere(Point(0, 0, 0), radius=2.5))
    viz(Plane(point=Point(0, 0, 3), normal=Direction(0, 0, 1)), opacity=0.25)

## Re-running the constructor

`Visualizer()` is a **singleton under Jupyter** — re-running the construction cell returns the same instance instead of trying to bind the port again, and clears the default scene (re-adding axes/grid per `add_default_axes` / `add_default_grid`).


In [ ]:
# Re-running this cell resets the default scene instead of accumulating.
viz = Visualizer()
viz.add(Point(1, 0, 1), color="#888888")
viz.show()

## Executed repeatedly

`show()` starts the server on the first call and renders inline. Re-running the cell (or calling `show()`/`display()` again) does **not** open a second viewer — it just flushes the latest state.


In [ ]:
viz(Point(4, 5, 6), color="#44ff44")
viz.show()  # no new viewer — just flushes the update

## Re-running a scene cell

`viz.scene(name)` is safe to re-run: re-running the *same* cell clears that scene (re-adding its axes/grid) before drawing again, so it never accumulates stale objects. A *different* cell that calls `viz.scene(name)` gets the existing scene without clearing, so you can keep building on it.


In [ ]:
# Re-running this cell clears "staging" first, then re-adds its content.
staging = viz.scene("staging")
staging.add(Sphere(Point(1, 0, 0), radius=1), opacity=0.8)
staging.show()

## Multiple scenes side by side

`display_row()` takes `(scene_handle, viewer_name)` tuples. The optional
`viewer_name` becomes a `?viewer=` label you can later target with
`navigate_to(target="viewer:<name>")` to drive one pane independently.


In [ ]:
overview = viz.scene("overview")
detail = viz.scene("detail")

overview.add(Sphere(Point(0, 0, 0), radius=3), opacity=0.2)
detail.add(Sphere(Point(2, 1, 0), radius=1), opacity=0.8)

# Label each pane so it can be addressed individually later.
viz.display_row((overview, "left"), (detail, "right"), height=400)


In [ ]:

# Later: switch only the "left" pane to the detail scene.
viz.navigate_to("detail", target="viewer:left")

## Cleanup

Stop the server when you are done to free the port.


In [ ]:
viz.stop_server()